# 01. Layer 1 — Agent / Tool / Model / Plugin

Layer 1 是「**單一 Agent 的組成**」，也是 ADK 最小可用單位。本 notebook 要做四件事：

1. 寫一個 **Tool**（function tool 形式）
2. 把 Tool 跟 **Model** 組成一個 **Agent**
3. 學會正確處理 reasoning model 的多段 Part 回應
4. 用 **Plugin（callback）** 加上 audit log，示範 ADK 的擴充機制

**情境：** 一個會查天氣的助理。

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import get_model, final_text, thought_parts

## 1. Tool — Agent 的手腳

`FunctionTool` 把一個 Python 函式包成 LLM 可呼叫的工具。三個來自函式的資訊會被帶給模型：

| 來源 | 變成什麼 |
|------|---------|
| 函式名 | tool name |
| type hints | 參數 schema（給模型決定怎麼填）|
| docstring | 工具描述（模型用來判斷何時要呼叫）|

下面用 mock data 模擬天氣 API。實務上這裡會是 `requests.get(...)` 或 SDK call。

In [2]:
from google.adk.tools import FunctionTool

_WEATHER_DB = {
    "taipei":  {"temp_c": 26, "condition": "多雲時晴", "humidity": 72},
    "tokyo":   {"temp_c": 18, "condition": "晴",       "humidity": 55},
    "kyoto":   {"temp_c": 16, "condition": "陰",       "humidity": 60},
    "new york":{"temp_c":  9, "condition": "小雨",     "humidity": 80},
    "paris":   {"temp_c": 12, "condition": "霧",       "humidity": 85},
}

def get_weather(city: str) -> dict:
    """Look up current weather for a city.

    Args:
        city: City name in English, e.g. 'Taipei', 'Tokyo'.
    """
    data = _WEATHER_DB.get(city.lower())
    if not data:
        return {"status": "error", "message": f"No data for {city}"}
    return {"status": "ok", "city": city, **data}

weather_tool = FunctionTool(func=get_weather)
print("Tool name:", weather_tool.name)
print("Tool desc:", weather_tool.description)

Tool name: get_weather
Tool desc: Look up current weather for a city.

Args:
    city: City name in English, e.g. 'Taipei', 'Tokyo'.


## 2. Agent = Model + Instruction + Tools

建一個 `LlmAgent`。重要參數：
- `model`：用 `get_model()` 拿到接好地端的實例
- `instruction`：角色定義 — 對 reasoning model 來說，**指令要直接、強硬**，模糊指令會讓它思考過久
- `tools`：可呼叫的工具清單

In [3]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

weather_agent = LlmAgent(
    name="weather_agent",
    model=get_model(),
    instruction=(
        "你是天氣助理。當使用者詢問城市天氣時，**一定要先呼叫 get_weather 工具**，"
        "再用一句繁體中文回答結果。如果工具回 error，就說『查不到該城市的資料』。"
    ),
    tools=[weather_tool],
)

## 3. 跑起來 — 並修掉「reasoning 洩漏」的顯示問題

00_setup 看到 Agent 回應像在自言自語？真相是 reasoning model 回的 final response 有**多個 Part**：

- `Part(thought=True)` → 模型的思考過程（給開發者除錯用）
- `Part(thought=None)` → 真正要給使用者的答案

我們在 `shared/runtime.py` 寫了 `final_text()` helper 來只取後者。這是顯示邏輯，不是 Plugin。

In [4]:
APP_NAME = "layer1"
USER_ID = "sean"

session_service = InMemorySessionService()
await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id="s1")
runner = Runner(agent=weather_agent, app_name=APP_NAME, session_service=session_service)

msg = types.Content(role="user", parts=[types.Part(text="東京現在天氣如何？")])

async for event in runner.run_async(user_id=USER_ID, session_id="s1", new_message=msg):
    if event.get_function_calls():
        for c in event.get_function_calls():
            print(f"  🔧 {c.name}({c.args})")
    if event.get_function_responses():
        for r in event.get_function_responses():
            print(f"  📦 {r.response}")
    if event.is_final_response():
        thoughts = thought_parts(event)
        if thoughts:
            print(f"\n💭 模型內心戲（{len(thoughts)} 段）：{thoughts[0].text[:120]}...")
        print(f"\n=== Agent 回覆 ===\n{final_text(event)}")

  🔧 get_weather({'city': 'Tokyo'})
  📦 {'status': 'ok', 'city': 'Tokyo', 'temp_c': 18, 'condition': '晴', 'humidity': 55}



💭 模型內心戲（1 段）：We have fetched Tokyo info: temp 18°C, condition 晴, humidity 55%. Need answer in one sentence Traditional Chinese.

Thus...

=== Agent 回覆 ===
目前東京天氣為晴朗，溫度約 18℃，濕度約 55%。


## 4. Plugin — 在 Agent 生命週期插入跨切關注點

Plugin（在 ADK 體現為各種 callback）讓你**不修改 Agent 本身**就能注入邏輯到生命週期的特定階段：

| Hook 點 | 用途 |
|---------|------|
| `before_model_callback` | 改 prompt、注入 context、token 計算前準備 |
| `after_model_callback` | 修飾回應、紀錄 token 用量、安全過濾 |
| `before_tool_callback` | 驗證 / 改參數 / 阻擋特定呼叫 |
| `after_tool_callback` | 紀錄結果、改回應、catching errors |

下面用 `before_tool_callback` 寫一個 **audit logger** — 記錄每次工具呼叫，是真實系統很常見的合規需求。

In [5]:
from typing import Any, Optional
from datetime import datetime
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext

AUDIT_LOG: list[dict] = []

def audit_tool_calls(
    tool: BaseTool,
    args: dict[str, Any],
    tool_context: ToolContext,
) -> Optional[dict]:
    """Append every tool invocation to AUDIT_LOG. Returning None lets the call proceed normally."""
    AUDIT_LOG.append({
        "ts": datetime.utcnow().isoformat(timespec="seconds"),
        "agent": tool_context.agent_name,
        "tool": tool.name,
        "args": dict(args),
    })
    return None  # 不阻擋呼叫

weather_agent_audited = LlmAgent(
    name="weather_agent_audited",
    model=get_model(),
    instruction=weather_agent.instruction,
    tools=[weather_tool],
    before_tool_callback=audit_tool_calls,
)

In [6]:
session_service2 = InMemorySessionService()
await session_service2.create_session(app_name=APP_NAME, user_id=USER_ID, session_id="s2")
runner2 = Runner(agent=weather_agent_audited, app_name=APP_NAME, session_service=session_service2)

for q in ["巴黎現在天氣？", "紐約呢？", "火星呢？"]:
    msg = types.Content(role="user", parts=[types.Part(text=q)])
    async for event in runner2.run_async(user_id=USER_ID, session_id="s2", new_message=msg):
        if event.is_final_response():
            print(f"Q: {q}\nA: {final_text(event)}\n")

print("=== AUDIT LOG ===")
for entry in AUDIT_LOG:
    print(entry)

/tmp/ipykernel_757032/2290287295.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat(timespec="seconds"),


Q: 巴黎現在天氣？
A: 巴黎目前氣温12℃，天氣為霧，濕度85%。



Q: 紐約呢？
A: 紐約目前氣溫9°C，天氣屬於小雨，濕度80%。



Q: 火星呢？
A: 查不到該城市的資料。

=== AUDIT LOG ===
{'ts': '2026-04-28T12:27:30', 'agent': 'weather_agent_audited', 'tool': 'get_weather', 'args': {'city': 'Paris'}}
{'ts': '2026-04-28T12:27:31', 'agent': 'weather_agent_audited', 'tool': 'get_weather', 'args': {'city': 'New York'}}
{'ts': '2026-04-28T12:27:33', 'agent': 'weather_agent_audited', 'tool': 'get_weather', 'args': {'city': 'Mars'}}


## 結論

你已經組裝出 Layer 1 的四個元件：

- **Tool** — `FunctionTool` 包裝 Python 函式
- **Model** — 透過 `LiteLlm` 接地端 / 雲端模型
- **Agent** — `LlmAgent` 把 instruction + model + tools 兜起來
- **Plugin** — `before_tool_callback`（或其他 hook）注入跨切關注點

**設計哲學**：Plugin 故意不寫進 Agent 本身，是因為 audit / monitoring / 安全過濾這些東西**不應該污染業務邏輯**。一個 Plugin 寫一次、所有 Agent 套用，這是 ADK 的「one register, global apply」設計。

下一站：`02_layer2_runtime.ipynb` — Runner / Session / Memory / Artifact 怎麼合作。